# 내부 점검용 노트북
# 반출용 코드로 사용할 때는 df.head(), sample_records 제거 필요

In [1]:
# 노트북에서 src/utils/io.py를 불러올 수 있게 경로 잡기
# read_table_flexible() 바로 쓰기
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.io import read_table_flexible

In [ ]:
# 환경 확인용
packages = ["pandas", "numpy", "geopandas", "shapely", "sklearn", "h3"]

for pkg in packages:
    try:
        module = __import__(pkg)
        print(f"{pkg}: OK / version={getattr(module, '__version__', 'unknown')}")
    except Exception as e:
        print(f"{pkg}: FAIL / {e}")

In [ ]:
# 파일 읽기 테스트용
# 파일 하나 경로 넣고 실제로 읽히는지, 인코딩/구분자가 뭐로 잡히는지, 컬럼이 뭐가 있는지 확인
# B013 / B042 / B024 / B021 / B075 후보 중 하나씩 테스트
test_path = Path("여기에_파일경로_입력.txt")  
# test_path = Path(r"D:/.../파일명.txt")

df, meta = read_table_flexible(test_path, nrows=1000)

print("읽기 성공")
print("encoding:", meta["encoding"])
print("sep:", repr(meta["sep"]))
print("shape:", df.shape)
print("columns:")
print(df.columns.tolist())

In [ ]:
#데이터 앞부분 확인 - 반출 시에 해당 출력 셀 삭제할 것!!
df.head(3)

In [ ]:
# 컬럼/결측/자료형 빠르게 보기
print(df.dtypes)
print("\n결측치 개수 상위 20개")
print(df.isna().sum().sort_values(ascending=False).head(20))

---

In [ ]:
# 좌표/공간 데이터 확인용
candidate_cols = [c for c in df.columns if any(k in c.upper() for k in ["X", "Y", "LON", "LAT", "COORD"])]
candidate_cols

In [ ]:
#geopandas import 및 geometry 생성 테스트
import geopandas as gpd
from shapely.geometry import Point

x_col = "X_COORD"   # 실제 컬럼명으로 수정
y_col = "Y_COORD"   # 실제 컬럼명으로 수정

sample_geo = df[[x_col, y_col]].dropna().head(100).copy()
sample_geo["geometry"] = sample_geo.apply(lambda r: Point(r[x_col], r[y_col]), axis=1)

gdf = gpd.GeoDataFrame(sample_geo, geometry="geometry", crs="EPSG:5179")  # 실제 좌표계에 맞게 수정
gdf.head()

# 좌표계(crs) 주의! 데이터 설명서 기반 : 
# B012 블록단위 지역경계도: EPSG:5181
# B024 블록단위 분기별 추정매출액의 블록 참조 shp: EPSG:5181
# B008/B010 SKT 유동인구 일부 좌표: EPSG:5179
# B009 KT 유동인구: EPSG:5186
# B013 버스정류장 좌표: EPSG:4326
# 파일 읽고 컬럼 보고, 좌표계도 같이 메모해야 함

In [ ]:
# 좌표계 변환 테스트
#나중에 H3 넣을 때 보통 WGS84 기준이 편해서 미리 변환해보는 셀
gdf_4326 = gdf.to_crs("EPSG:4326")
gdf_4326.head()

In [ ]:
# H3 테스트 셀 - h3 import가 되는 경우에만
import h3

In [ ]:
gdf_4326["lat"] = gdf_4326.geometry.y
gdf_4326["lng"] = gdf_4326.geometry.x

gdf_4326["h3_r8"] = gdf_4326.apply(
    lambda r: h3.latlng_to_cell(r["lat"], r["lng"], 8),
    axis=1
)

gdf_4326[["lat", "lng", "h3_r8"]].head()

---

In [ ]:
# scripts/01_scan_dataset.py

from pathlib import Path
import json
import pandas as pd

from src.utils.io import read_table_flexible
from src.config import OUTPUT_DIR

def summarize_table(path: str):
    df, meta = read_table_flexible(path, nrows=1000)

    summary = {
        "file": str(path),
        "encoding": meta["encoding"],
        "sep": meta["sep"],
        "shape_preview": [int(df.shape[0]), int(df.shape[1])],
        "columns": df.columns.tolist(),
        "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
        "null_counts_top20": df.isna().sum().sort_values(ascending=False).head(20).to_dict(),
        "sample_records": df.head(3).to_dict(orient="records"),
    }
    return summary

if __name__ == "__main__":
    import sys

    if len(sys.argv) < 2:
        raise SystemExit("사용법: python scripts/01_scan_dataset.py <파일경로>")

    out_dir = OUTPUT_DIR / "dataset_scans"
    out_dir.mkdir(parents=True, exist_ok=True)

    target = sys.argv[1]
    summary = summarize_table(target)

    out_path = out_dir / (Path(target).stem + "_summary.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"[완료] {out_path}")

In [ ]:
# scripts/02_check_env.py

def try_import(name):
    try:
        module = __import__(name)
        return True, getattr(module, "__version__", "unknown")
    except Exception as e:
        return False, str(e)

if __name__ == "__main__":
    packages = ["pandas", "numpy", "geopandas", "shapely", "sklearn", "h3"]
    for pkg in packages:
        ok, info = try_import(pkg)
        print(f"{pkg}: {'OK' if ok else 'FAIL'} / {info}")